# EOA - Notebook de Exploração Experimental

Este notebook acompanha a avaliação prática do framework **EOA (*Explanation-Oriented Agents*)** usando dados reais do sistema hidráulico da ZeMA.

- **Parte 1 (Demonstração Guiada):** um passo a passo direto mostrando inicialização com banco SQLite (`eoa.db`), registro de ferramentas (`tools`) no CMA e AA, explicação no modo Zero-ML, listagem de incidentes, explicação de predição e criação/edição/remoção de agentes.
- **Parte 2 (Agora é a sua vez):** espaço livre com sugestões para você testar o framework do seu jeito, criar células novas e avaliar a experiência de uso.

> Consulte o [`GUIDE.md`](./GUIDE.md) sempre que quiser conferir parâmetros e assinaturas completas.

---
## Parte 1: Demonstração Guiada

### 1. Setup do ambiente
Carrega o arquivo `.env` com a chave da API do Gemini e define uma função auxiliar simples para imprimir os resultados de forma legível.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Carrega automaticamente a chave do arquivo .env desta pasta
load_dotenv(Path(".env"))
print(f"GEMINI_API_KEY carregada: {bool(os.environ.get('GEMINI_API_KEY'))}")


def mostrar_resposta(res) -> None:
    """Exibe a narrativa gerada, a nota do validador e o rastro dos agentes."""
    print("=" * 80)
    print("EXPLICAÇÃO GERADA:")
    print("-" * 80)
    print(res.narrative)
    print("-" * 80)
    print(
        f"Nota Validador: {res.validation_score}/10.0 | "
        f"Aprovado: {res.validation_passed} | "
        f"Tentativas: {res.attempts_used} | "
        f"Fallback: {res.fallback_used}"
    )
    print("-" * 80)
    for step in res.execution_trace:
        print(f"[{step.agent_name:<23}] {step.status:<10} | {step.execution_time_ms:7.1f} ms | {step.summary}")
    print("=" * 80)

GEMINI_API_KEY carregada: True


### 2. Imports e inicialização do `eoa` com `eoa.db`
Importamos diretamente as classes do EOA e as funções do módulo `tools.py`, criamos um `SQLiteSessionStore("eoa.db")` para persistência local e inicializamos a camada `eoa` com o construtor `ExplanationLayer`.

In [2]:
from eoa import (
    DomainContext,
    ExplanationLayer,
    PredictionArtifact,
    SpecializedAgent,
    SQLiteSessionStore,
    XAIArtifact,
)
from tools import (
    get_baseline_comparison,
    get_incident,
    get_sensor_readings,
    list_incidents,
    load_domain_context,
)

# Cria o banco SQLite local 'eoa.db' para gravar sessões, mensagens e notas de validação
store = SQLiteSessionStore("eoa.db")

# Inicializa a camada usando o construtor direto com gemini-3.6-flash.
# Passamos auto_create_agents=True para já instanciar os 4 agentes padrão (CMA, AA, EA, VA).
eoa = ExplanationLayer(
    provider="gemini",
    model="gemini-3.6-flash",
    session_store=store,
    auto_create_agents=True,
)
eoa.inspect()

EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          0        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Built-in
validator          Quality Gate Auditor     0        Built-in


### 3. Registrando tools no `ContextManagerAgent` (CMA)
Em vez de passar o contexto manualmente em toda chamada, registramos `load_domain_context`, `get_sensor_readings` e `get_baseline_comparison` como ferramentas do CMA. Assim, ele mesmo consulta a documentação e os dados dos sensores quando necessário.

In [3]:
cma = eoa.get_agent("context_manager")

# load_domain_context(): carrega a documentação do sistema hidráulico ZeMA e o dicionário dos 14 sensores
cma.register_tool(load_domain_context)

# get_sensor_readings(incident_id): retorna as médias dos 14 sensores físicos do ciclo
cma.register_tool(get_sensor_readings)

# get_baseline_comparison(incident_id): compara o ciclo com a média e percentis de operação 100% saudável
cma.register_tool(get_baseline_comparison)

# Confira na coluna 'Tools' que o context_manager agora possui 3 ferramentas registradas
eoa.inspect()

EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          3        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Built-in
validator          Quality Gate Auditor     0        Built-in


### 4. Explicação no Modo Zero-ML (CMA buscando o contexto via tool)
Chamamos `eoa.explain(...)` apenas com a pergunta e o `session_id`. Sem passarmos nenhum `context=` nem predição na mão, o `ContextManagerAgent` chama `load_domain_context` via tool, enquanto o `ArtifactsAgent` detecta ausência de ML e pula a execução (`0.0 ms`).

In [4]:
eoa.clear_session("demo_zero_ml")

res_zero_ml = eoa.explain(
    session_id="demo_zero_ml",
    user_query=(
        "Explique em termos práticos como funciona o sistema hidráulico ZeMA, "
        "quais são os grupos de sensores físicos monitorados e o que significam "
        "os três códigos de condição do refrigerador (Cooler condition / %)."
    ),
)

mostrar_resposta(res_zero_ml)

Orchestrator caught exception (ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 11h37m43.023026495s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on F

EXPLICAÇÃO GERADA:
--------------------------------------------------------------------------------
[FALLBACK DETERMINÍSTICO ATIVADO / DEGRADAÇÃO GRACIOSA]
Motivo Técnico: Exceção interceptada (ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 11h37m43.023026495s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/d

### 5. Consultando os incidentes disponíveis com `list_incidents`
Antes de pedir a explicação de uma predição, usamos `list_incidents()` para inspecionar os 9 casos de teste pré-calculados pelo Random Forest + TreeSHAP.

In [5]:
# Como usar as ferramentas disponíveis em tools.py:
# - list_incidents()                     -> lista todos os 9 incidentes curados (códigos 3, 20 e 100)
# - list_incidents(condition_code=3)     -> filtra apenas incidentes em falha crítica (3%)
# - load_domain_context()                -> carrega documentação do ZeMA e referência dos sensores
# - get_incident(incident_id)            -> retorna {'prediction': ..., 'xai': ...} do incidente
# - get_sensor_readings(incident_id)     -> retorna as leituras médias dos 14 sensores físicos
# - get_baseline_comparison(incident_id) -> retorna delta e z-score vs. operação saudável (100%)

incidentes = list_incidents()

print(f"{'ID':<4} {'Ciclo':<7} {'Condição Prevista':<30} {'Conf.':<7} {'Stable':<7} {'Top 3 TreeSHAP'}")
print("-" * 100)
for inc in incidentes:
    top3 = ", ".join(f"{k}: {v:+.4f}" for k, v in inc["top_3_shap_features"].items())
    print(
        f"{inc['incident_id']:<4} {inc['cycle_id']:<7} "
        f"{inc['predicted_condition_label']:<30} {inc['confidence']:<7.2f} "
        f"{inc['stable_flag']:<7} {top3}"
    )

ID   Ciclo   Condição Prevista              Conf.   Stable  Top 3 TreeSHAP
----------------------------------------------------------------------------------------------------
1    2       3% (close to total failure)    0.50    1       FS2: +0.1935, PS4: +0.1051, PS6: -0.0762
2    241     3% (close to total failure)    1.00    0       PS6: +0.1670, PS5: +0.1557, TS4: +0.1340
3    492     3% (close to total failure)    1.00    0       PS6: +0.1588, PS5: +0.1482, TS4: +0.1174
4    731     3% (close to total failure)    0.98    0       PS6: +0.1815, PS5: +0.1574, TS4: +0.1327
5    737     20% (reduced efficiency)       0.58    1       VS1: +0.2067, PS5: -0.0599, PS3: +0.0444
6    1067    20% (reduced efficiency)       1.00    1       TS4: +0.1458, PS5: +0.1385, PS6: +0.1314
7    1458    20% (reduced efficiency)       1.00    0       TS4: +0.1347, PS5: +0.1302, PS6: +0.1277
8    1467    100% (full efficiency)         0.73    1       TS4: +0.2189, PS4: +0.0351, PS2: +0.0303
9    2202    100

### 6. Pedindo a explicação de uma predição
Registramos `get_incident` no `ArtifactsAgent` e passamos `incident_id=2` (um caso de falha crítica `3%` em regime térmico estável). O framework dispara automaticamente as tools do `artifacts` e do `context_manager` para reunir predição, TreeSHAP, leituras e desvios da linha de base.

In [ ]:
# Registra a tool de busca de predição + XAI no ArtifactsAgent
eoa.get_agent("artifacts").register_tool(get_incident)

eoa.clear_session("demo_incidente_2")

# Agora basta passar incident_id=2: AA e CMA buscam predição, SHAP, contexto, sensores e baseline via tools
res_predicao = eoa.explain(
    incident_id=2,
    session_id="demo_incidente_2",
    user_query=(
        "Por que o modelo classificou o incidente 2 como 3% (próximo à falha total)? "
        "Relacione os principais atributos do TreeSHAP com as leituras reais dos sensores "
        "e o quanto eles desviaram da operação saudável."
    ),
)

mostrar_resposta(res_predicao)

### 7. Criando um agente, sobrescrevendo o prompt, executando e removendo
Demonstramos a extensão dinâmica: criamos um `SpecializedAgent`, aplicamos `override_agent_prompt` para definir suas instruções, rodamos um exemplo no Incidente 5 e depois removemos o agente com `remove_agent`.

In [ ]:
# 1. Cria um agente especialista (na v0.1.0 o construtor exige um prompt inicial não vazio).
#    Passamos get_sensor_readings em tools=[] para que ele acesse stable_flag e TS1-TS4 em paralelo.
agente_regime = SpecializedAgent(
    name="analista_regime_termico",
    role="Especialista em Estabilidade Térmica",
    prompt="Analise o incidente.",
    tools=[get_sensor_readings],
)
eoa.register_agent(agente_regime)

# 2. Sobrescreve o prompt do agente recém-criado usando os dados obtidos pela tool ({tool_outputs})
eoa.override_agent_prompt(
    "analista_regime_termico",
    """Você é um Especialista em Estabilidade Térmica de Sistemas Hidráulicos.
Analise os dados de telemetria coletados ({tool_outputs}).
Em no máximo 3 linhas diretas:
1. Informe se o ciclo está em regime estável (stable_flag=0) ou em transição (stable_flag=1).
2. Explique se os sensores de temperatura (TS1 a TS4) confirmam aquecimento consistente ou fase transitória.
""",
)

# Confira o novo agente e a origem do prompt ('Inline (str)') na tabela
eoa.inspect()

# 3. Executa um exemplo com o agente especialista participando do pipeline
eoa.clear_session("demo_incidente_5")
res_especialista = eoa.explain(
    incident_id=5,
    session_id="demo_incidente_5",
    user_query="Avalie a condição do refrigerador no incidente 5 considerando a estabilidade térmica do ciclo.",
)
mostrar_resposta(res_especialista)

# 4. Remove o agente especialista ao final da demonstração e confirma a topologia original
eoa.remove_agent("analista_regime_termico")
eoa.inspect()

# Confira as sessões gravadas no banco SQLite 'eoa.db'
print("Sessões salvas em eoa.db:", store.list_sessions())

---
## Parte 2: Agora é a sua vez!

A célula abaixo é um espaço aberto para você experimentar o EOA livremente. Fique à vontade para editar o código, descomentar as sugestões ou **criar quantas células novas quiser**.

Você pode explorar, sem se limitar a isso:
- **Criar e registrar novos agentes especialistas:** `seu_agente = SpecializedAgent(...)` e `eoa.register_agent(seu_agente)`.
- **Editar prompts de agentes existentes:** `eoa.override_agent_prompt("explainer", ...)` ou restaurar com `eoa.reset_agent_prompt("explainer")`.
- **Criar suas próprias funções Python e registrá-las como `tools`:** `eoa.get_agent("context_manager").register_tool(sua_funcao)`.
- **Testar outros incidentes (`1` a `9`), modo Push (`prediction=` / `xai=`), conversas multi-turno com `session_id` (auditando o `eoa.db` com `store.get_validation_logs(...)`), ou remover agentes (`eoa.remove_agent(...)`).**

In [ ]:
# Espaço livre para você experimentar o EOA (crie novas células abaixo se quiser).
# Alguns exemplos rápidos que você pode descomentar e adaptar:

# --- Sugestão 1: Criar e registrar um agente especialista próprio ---
# seu_agente = SpecializedAgent(
#     name="auditor_pressao",
#     role="Especialista em Pressão Hidráulica",
#     prompt="Analise exclusivamente os sensores PS1 a PS6 em {context_summary} para o incidente {query_params}.",
# )
# eoa.register_agent(seu_agente)

# --- Sugestão 2: Criar uma nova tool Python e registrar no ContextManagerAgent ---
# def resumo_temperaturas(incident_id: int) -> dict:
#     """Retorna apenas as temperaturas TS1 a TS4 e seus desvios da linha de base."""
#     comp = get_baseline_comparison(incident_id)["comparisons"]
#     return {k: comp[k] for k in ("TS1", "TS2", "TS3", "TS4")}
#
# eoa.get_agent("context_manager").register_tool(resumo_temperaturas)

# --- Sugestão 3: Customizar o prompt do ExplainerAgent (ou restaurar depois) ---
# eoa.override_agent_prompt(
#     "explainer",
#     "Responda em 3 tópicos objetivos para um operador de chão de fábrica.\nEvidências: {artifacts}\nContexto: {domain_context}\nPergunta: {user_query}",
# )
# # eoa.reset_agent_prompt("explainer")

# --- Sugestão 4: Rodar outro incidente (1 a 9), conversa multi-turno e auditar o eoa.db ---
# res = eoa.explain(
#     incident_id=1,
#     session_id="meu_teste_01",
#     user_query="Por que a confiança do modelo ficou em apenas 50% no incidente 1?",
# )
# mostrar_resposta(res)
#
# res_continuacao = eoa.explain(
#     session_id="meu_teste_01",
#     user_query="Qual foi o valor exato do sensor FS2 nesse ciclo?",
# )
# mostrar_resposta(res_continuacao)
#
# print("Logs de validação no eoa.db:", store.get_validation_logs("meu_teste_01"))

eoa.inspect()